In [ ]:
%pwd

'/content'

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')  # follow the auth link

# 2) Pick a folder in Drive to keep repos
repo_root = "/content/drive/MyDrive/ColabRepos"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3) Clone directly into Drive
!mkdir -p "{repo_root}"
!git clone https://github.com/XuZR3x/Retrieval_Reasoning_Clinical_Trial_Generation.git "{repo_root}/REPO"

In [ ]:
%mkdir /content/drive/MyDrive/ColabRepos/REPO/data/raw
%cd /content/drive/MyDrive/ColabRepos/REPO/data/raw
!wget https://clinicaltrials.gov/AllPublicXML.zip
!unzip AllPublicXML.zip -d ../trials

Streaming output truncated to the last 5000 lines.
  inflating: ../trials/NCT0503xxxx/NCT05036733.xml  
  inflating: ../trials/NCT0041xxxx/NCT00410033.xml  
  inflating: ../trials/NCT0135xxxx/NCT01359033.xml  
  inflating: ../trials/NCT0372xxxx/NCT03721133.xml  
  inflating: ../trials/NCT0409xxxx/NCT04095533.xml  
  inflating: ../trials/NCT0444xxxx/NCT04440033.xml  
  inflating: ../trials/NCT0102xxxx/NCT01026233.xml  
  inflating: ../trials/NCT0338xxxx/NCT03387033.xml  
  inflating: ../trials/NCT0256xxxx/NCT02565433.xml  
  inflating: ../trials/NCT0052xxxx/NCT00524433.xml  
  inflating: ../trials/NCT0165xxxx/NCT01651533.xml  
  inflating: ../trials/NCT0623xxxx/NCT06239233.xml  
  inflating: ../trials/NCT0240xxxx/NCT02401633.xml  
  inflating: ../trials/NCT0675xxxx/NCT06754033.xml  
  inflating: ../trials/NCT0457xxxx/NCT04577833.xml  
  inflating: ../trials/NCT0122xxxx/NCT01225133.xml  
  inflating: ../trials/NCT0661xxxx/NCT06612333.xml  
  inflating: ../trials/NCT0650xxxx/NCT06508333.x

In [ ]:
import os
import json
import time
import random
import itertools
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from tqdm import tqdm
import xml.etree.ElementTree as ET
import re
import tiktoken

In [ ]:
import os

# Set your API key directly
os.environ["OPENAI_API_KEY"] = 'ADD-YOUR-API-KEY'

In [ ]:
def list_all_files_in_subfolders(parent_directory):

    file_list = []

    try:
        for root, dirs, files in os.walk(parent_directory):
            for file in files:
                if file.endswith('.xml'):
                    file_name_without_extension = os.path.splitext(file)[0]
                    file_list.append(file_name_without_extension)
    except Exception as e:
        print(f"An error occurred: {e}")

    return file_list

def process_all():

    parent_directory = '/content/drive/MyDrive/ColabRepos/REPO/data/trials'
    output_file = '/content/drive/MyDrive/ColabRepos/REPO/data/trials/all_trials.txt'
    file_list = list_all_files_in_subfolders(parent_directory)

    with open(output_file, 'w') as f:
        for file_name in file_list:
            f.write(file_name + '\n')

In [ ]:
process_all()

In [ ]:
# Load the CSV file
csv_path = '/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/trial_outcomes_v1.csv'
df = pd.read_csv(csv_path)

# Load the mapping from the txt file
mapping_path = '/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/outcome2label.txt'
mapping = {}
with open(mapping_path, 'r') as file:
    for line in file:
        outcome, label = line.strip().split('\t')
        mapping[outcome] = int(label)

# Apply the mapping to the DataFrame
df['trialOutcome'] = df['trialOutcome'].map(mapping)

# Rename the second column to 'label'
df.rename(columns={'trialOutcome': 'label'}, inplace=True)

# Remove rows with label -1
df = df[df['label'] != -1]

all_trials_path = '/content/drive/MyDrive/ColabRepos/REPO/data/trials/all_trials.txt'
df_trials = pd.read_csv(all_trials_path, header=None, names=['studyid'])
df_trials = df_trials.sort_values(by='studyid').reset_index(drop=True)

df_filtered = pd.merge(df, df_trials, on='studyid', how='inner')

# Save the updated DataFrame to a new CSV file
output_path = '/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/filtered_trial_outcomes_with_labels.csv'
df_filtered.to_csv(output_path, index=False)

In [ ]:
file_path = '/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/filtered_trial_outcomes_with_labels.csv'
df = pd.read_csv(file_path)

def element_to_dict(el):

    children = list(el)
    if not children:
        return el.text
    result = {}
    for child in children:
        child_dict = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_dict)
        else:
            result[child.tag] = child_dict

    return result

def xml_to_dict(element):

    return {element.tag: element_to_dict(element)}

def read_xml_file(file_path):

    tree = ET.parse(file_path)
    root = tree.getroot()
    return xml_to_dict(root)

def extract_intervention_name(study_id):

    xml_path = f'/content/drive/MyDrive/ColabRepos/REPO/data/trials/{study_id[:7]}xxxx/{study_id}.xml'
    xml_dict = read_xml_file(xml_path)
    xml_string = json.dumps(xml_dict)
    pattern = r'"intervention_name":\s*"([^"]*)"'
    match = re.search(pattern, xml_string)

    if match:
        intervention_name = match.group(1).lower()
        return intervention_name

def num_tokens_from_string(str):
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    num_tokens = len(encoding.encode(str))
    return num_tokens

df['intervention_name'] = df['studyid'].apply(extract_intervention_name)

drug_name_file_path = '/content/drive/MyDrive/ColabRepos/REPO/data/raw/drugbank vocabulary.csv'
drug_name_df = pd.read_csv(drug_name_file_path)

drug_name_list = drug_name_df['Common name'].tolist()
drug_name_list_lowercased = [name.lower() for name in drug_name_list]
drug_set = set(drug_name_list_lowercased )

df_filtered = df[df['intervention_name'].isin(drug_set)]

label_counts = df_filtered.groupby(['intervention_name', 'label']).size().unstack(fill_value=0)

# Filter for intervention names where both label 0 and label 1 have at least 3 occurrences
valid_interventions = label_counts[(label_counts[0] >= 3) & (label_counts[1] >= 3)].index

# Filter the original DataFrame based on the valid intervention names
df_filtered_final = df_filtered[df_filtered['intervention_name'].isin(valid_interventions)]

# Group by intervention_name and label, then count occurrences in df_filtered_final
label_counts_final = df_filtered_final.groupby(['intervention_name', 'label']).size().unstack(fill_value=0)

# Display the counts for each intervention name
print(label_counts_final)

intervention_names = label_counts_final.index.tolist()
print(intervention_names)

label               0   1
intervention_name        
abatacept          12  21
abemaciclib         3   7
adalimumab          6  61
afatinib            9   7
aflibercept         9  13
...                ..  ..
vorinostat         21  13
vortioxetine        6  10
warfarin            5   6
ziprasidone         9   9
zoledronic acid    14  19

[245 rows x 2 columns]
['abatacept', 'abemaciclib', 'adalimumab', 'afatinib', 'aflibercept', 'aldesleukin', 'alemtuzumab', 'alfuzosin', 'alisertib', 'aliskiren', 'anakinra', 'anastrozole', 'apixaban', 'aprepitant', 'aripiprazole', 'arsenic trioxide', 'asenapine', 'atazanavir', 'atezolizumab', 'atomoxetine', 'atorvastatin', 'auy922', 'axitinib', 'azacitidine', 'azithromycin', 'belimumab', 'belinostat', 'bendamustine', 'bevacizumab', 'bicalutamide', 'bivalirudin', 'bortezomib', 'bosentan', 'botulinum toxin type a', 'brentuximab vedotin', 'brexpiprazole', 'bupivacaine', 'buprenorphine', 'cabazitaxel', 'cabozantinib', 'candesartan', 'candesartan cilexetil',

In [ ]:
# ===== Build df_all and df_ff from your data/CT.gov XML =====
import os, re, json, xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
from functools import lru_cache
from tqdm import tqdm

# ---------- Paths ----------
BASE_CSV = "/content/drive/MyDrive/ColabRepos/REPO/data/IQVIA/filtered_trial_outcomes_with_labels.csv"
TRIALS_DIR = "/content/drive/MyDrive/ColabRepos/REPO/data/trials"   # where {studyid[:7]}xxxx/{studyid}.xml lives
DERIVED_DIR = Path("/content/drive/MyDrive/ColabRepos/REPO/data/derived")
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

DF_ALL_PATH = DERIVED_DIR / "df_all.parquet"
DF_FF_PATH  = DERIVED_DIR / "df_ff.parquet"
INTERV_LIST_PATH = DERIVED_DIR / "correct_intervention_list.json"
TEXT_CACHE_PATH  = DERIVED_DIR / "trial_text_cache.parquet"  # speeds up re-runs

# ---------- Helpers to read/clean XML ----------
def element_to_dict(el):
    children = list(el)
    if not children:
        return el.text
    result = {}
    for child in children:
        child_dict = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_dict)
        else:
            result[child.tag] = child_dict
    return result

def xml_to_dict(element):
    return {element.tag: element_to_dict(element)}

def read_xml_file(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    return xml_to_dict(root)

def _get_nested(d, keys):
    cur = d
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return None
    return cur

LEAK_RE = re.compile(r"(overall\s*status|why\s*stop(ped)?|success(ful|fully)?|fail(ed|ure)?)",
                     flags=re.IGNORECASE)

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    s = LEAK_RE.sub("[REDACTED]", s)
    return s

@lru_cache(maxsize=200000)
def load_trial_text_from_xml(studyid: str) -> str:
    xml_path = f"{TRIALS_DIR}/{studyid[:7]}xxxx/{studyid}.xml"
    if not os.path.exists(xml_path):
        return ""
    try:
        x = read_xml_file(xml_path)
        cs = x.get("clinical_study", {})
        parts = []
        # Common CT.gov fields to concatenate (add/remove as you like)
        fields = [
            ("brief_title",),
            ("official_title",),
            ("brief_summary", "textblock"),
            ("detailed_description", "textblock"),
            ("eligibility", "criteria", "textblock"),
            ("primary_outcome", "measure"),
            ("primary_outcome", "description"),
            ("secondary_outcome", "measure"),
            ("secondary_outcome", "description"),
        ]
        for path in fields:
            val = _get_nested(cs, path)
            if val is None:
                continue
            if isinstance(val, list):
                for v in val:
                    if isinstance(v, str) and v.strip():
                        parts.append(v.strip())
            elif isinstance(val, str):
                if val.strip():
                    parts.append(val.strip())
            elif isinstance(val, dict):
                tb = val.get("textblock")
                if isinstance(tb, str) and tb.strip():
                    parts.append(tb.strip())
        text = "\n\n".join(parts)
        return clean_text(text)
    except Exception:
        return ""

# ---------- Load base CSV & build intervention names if needed ----------
df = pd.read_csv(BASE_CSV)

def extract_intervention_name(study_id):
    xml_path = f"{TRIALS_DIR}/{study_id[:7]}xxxx/{study_id}.xml"
    if not os.path.exists(xml_path):
        return None
    try:
        xd = read_xml_file(xml_path)
        xml_string = json.dumps(xd)
        m = re.search(r'"intervention_name":\s*"([^"]*)"', xml_string)
        if m:
            return m.group(1).lower()
    except Exception:
        return None
    return None

if "intervention_name" not in df.columns or df["intervention_name"].isna().all():
    tqdm.pandas(desc="Extracting intervention_name")
    df["intervention_name"] = df["studyid"].progress_apply(extract_intervention_name)

# Map numeric labels to strings expected downstream
df["label_str"] = df["label"].map({1: "Success", 0: "Failure"})

# ---------- Build/Reuse text cache to speed up ----------
if TEXT_CACHE_PATH.exists():
    text_cache = pd.read_parquet(TEXT_CACHE_PATH)
    cache_map = dict(zip(text_cache["studyid"], text_cache["text"]))
else:
    cache_map = {}

def get_text_cached(studyid: str) -> str:
    if studyid in cache_map:
        return cache_map[studyid]
    txt = load_trial_text_from_xml(studyid)
    cache_map[studyid] = txt
    return txt

tqdm.pandas(desc="XML → text")
df["text"] = df["studyid"].progress_apply(get_text_cached)

# Persist cache
pd.DataFrame({"studyid": list(cache_map.keys()), "text": list(cache_map.values())}).to_parquet(TEXT_CACHE_PATH, index=False)

# ---------- Create df_all ----------
df_all = df.copy()
# Drop rows with missing key fields
df_all = df_all.dropna(subset=["intervention_name","label_str","text"])
# length filter (tweak as needed)
df_all = df_all[df_all["text"].str.len() > 50].reset_index(drop=True)

print("df_all size:", len(df_all))
print(df_all[["studyid","intervention_name","label_str"]].head())

# ---------- Choose interventions that can support K-shot per label ----------
# 3 for strict 3-shot.
min_per_label = 3

counts = (df_all.groupby(["intervention_name","label_str"])
          .size().unstack(fill_value=0))

ok = (counts.get("Success", 0) >= min_per_label) & (counts.get("Failure", 0) >= min_per_label)
correct_intervention_list = counts[ok].index.tolist()

print(f"Interventions with ≥{min_per_label} per label:", len(correct_intervention_list))

# ---------- Create df_ff (few-shot feasible subset) ----------
df_ff = df_all[df_all["intervention_name"].isin(correct_intervention_list)].reset_index(drop=True)

print("df_ff size:", len(df_ff))
print(df_ff["intervention_name"].value_counts().head())

# ---------- Save for reuse ----------
df_all.to_parquet(DF_ALL_PATH, index=False)
df_ff.to_parquet(DF_FF_PATH, index=False)
with open(INTERV_LIST_PATH, "w") as f:
    json.dump(correct_intervention_list, f)

print(f"\nSaved:\n- df_all -> {DF_ALL_PATH}\n- df_ff -> {DF_FF_PATH}\n- correct_intervention_list -> {INTERV_LIST_PATH}")


XML → text: 100%|██████████| 26768/26768 [00:00<00:00, 442420.81it/s]


df_all size: 26391
       studyid         intervention_name label_str
0  NCT00000172               galantamine   Success
1  NCT00000173                 donepezil   Success
2  NCT00000174              rivastigmine   Failure
3  NCT00000390  imipramine hydrochloride   Success
4  NCT00000419      premarin and provera   Failure
Interventions with ≥3 per label: 314
df_ff size: 7507
intervention_name
placebo        514
bevacizumab    213
rituximab      141
docetaxel      125
cetuximab      104
Name: count, dtype: int64

Saved:
- df_all -> /content/drive/MyDrive/ColabRepos/REPO/data/derived/df_all.parquet
- df_ff -> /content/drive/MyDrive/ColabRepos/REPO/data/derived/df_ff.parquet
- correct_intervention_list -> /content/drive/MyDrive/ColabRepos/REPO/data/derived/correct_intervention_list.json


In [ ]:
# =========================================================
# Retrieval–Reasoning Generator: Baseline, Variants, Ablations (separate runs)
# - EXACT prompts preserved
# - Plain text reports synthetic_XXXXXX.txt
# - labels.txt contains 0/1 per line (append; resume-safe)
#
# Requires: df_ff (or df_all), correct_intervention_list
# df_ff columns: 'intervention_name', 'text', and either 'label' (0/1) or 'label_str' ('Success'/'Failure')
# =========================================================

from __future__ import annotations
import os, re, random
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import pandas as pd
from string import Formatter

try:
    import tiktoken
except Exception:
    tiktoken = None

from openai import OpenAI

# ----------------------------
# Config
# ----------------------------
MODEL = "gpt-4o-mini"
TEMPERATURE = 1.0
TOKEN_BUDGET_EXAMPLES = 120_000
N_EXAMPLES = 3
RNG = random.Random()   # unseeded → nondeterministic label counts

# ----------------------------
# Prompts
# ----------------------------
REASON_CTX = (
    "You are now a medical expert in the clinical area. "
    "You are given information of a medical intervention, and three clinical "
    "trial reports of it, either all successful or all failed. "
    "You are asked to analyze these input and write reasons resulting the trials' success/failure. "
    "Your writing style must be consistent within the clinical study. "
    "You must ensure that your language is precise, technical, and formal."
)
REASON_FMT = (
    "Your output should strictly follow the following format: "
    "\n 1. (...) \n 2. (...) \n 3. (...) \n 4. (...) \n 5. (...), "
    "with (...) being the reasons you write."
)
REASON_GEN = "Write 5 reasons leading {intervention} to {label} in these trials. Remember to make sure that your language is precise, technical, and formal. Be creative and write unique reasons."
REASON_DIVERSITY = "Can you provide something more diverse compared to the previously generated reasons?"

GEN_CTX = (
    "You are now a medical expert in the clinical area. "
    "You are provided with reasons of a medical intervention that might lead to "
    "the success/failure of a clinical trial, with three real clinical trials that are either all successful/failed. "
    "You are asked to write a clinical trial for the intervention of the same label (failure/success). "
    "Your writing style must be consistent with the real clinical trial examples. "
    "You must ensure that your language is precise, technical, and formal."
)
GEN_CONSTRAINT = (
    "Your style of output should be strictly similar to the xml-like format of the provided three clinical trial examples, "
    "but you cannot simply modify or rewrite them. "
    "The name of the intervention must be {intervention}, and you must refer to the reasons when writing clinical trials."
)
GEN_REQ = "Write a report of a {label} clinical trial of {intervention}. Remember to make sure that your language is precise, technical, and formal. Be creative and write unique reports."
GEN_DIVERSITY = "Can you provide something more diverse compared to the previously generated trials?"

# Internal aliases
reason_context_prompt = REASON_CTX
reason_format_prompt = REASON_FMT
reason_data_generation_prompt = REASON_GEN
reason_diversity_prompt = REASON_DIVERSITY
context_prompt = GEN_CTX
constraint_prompt = GEN_CONSTRAINT
data_generation_prompt = GEN_REQ
diversity_prompt = GEN_DIVERSITY
reason_prompt = "Here are five reasons that could lead to the {type} of clinical trials of {name}: {reasons}"

# ----------------------------
# OpenAI client
# ----------------------------
client = OpenAI()  # expects OPENAI_API_KEY

# ----------------------------
# Variant config (for prompt sensitivity)
# ----------------------------
@dataclass
class VariantConfig:
    label_distribution: str = "same"              # "same", "mixed_head", "mixed_tail"
    example_ordering:   str = "as_is"             # "as_is", "shuffled"
    class_name_scheme:  str = "success_failure"   # "success_failure", "positive_negative"

# ----------------------------
# Label lexicon by scheme
# ----------------------------
def make_label_lexicon(scheme: str, y: int) -> Dict[str, str]:
    if scheme == "positive_negative":
        label_title = "Positive" if y == 1 else "Negative"
        type_noun   = "positive outcome" if y == 1 else "negative outcome"
        header_token = "Positive" if y == 1 else "Negative"
    else:
        label_title = "Success" if y == 1 else "Failure"
        type_noun   = "success" if y == 1 else "failure"
        header_token = "Successful" if y == 1 else "Failed"
    return dict(label_title=label_title, type_noun=type_noun, header_token=header_token)

# ----------------------------
# Utilities
# ----------------------------
def num_tokens_from_string(s: str) -> int:
    if tiktoken is None:
        return max(1, len(s) // 4)
    try:
        enc = tiktoken.encoding_for_model(MODEL)
    except Exception:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(s))

def _format_partial(template: str, **kwargs) -> str:
    fields = {fname for _, fname, _, _ in Formatter().parse(template) if fname}
    use = {k: v for k, v in kwargs.items() if k in fields}
    return template.format(**use) if use else template

# ----------------------------
# Data context
# ----------------------------
@dataclass
class DataContext:
    df_ff: pd.DataFrame
    valid_interventions: List[str]
    retrieval_used: bool

def _normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "label" not in df.columns:
        if "label_str" in df.columns:
            df["label"] = df["label_str"].map(
                lambda s: 1 if str(s).strip().lower() in {"success", "successful", "positive"} else 0
            )
        else:
            raise ValueError("df_ff must contain either 'label' or 'label_str'.")
    for col in ["intervention_name", "text", "label"]:
        if col not in df.columns:
            raise ValueError(f"df_ff missing required column: {col}")
    return df[["intervention_name", "text", "label"]]

def build_data_context(
    df_ff: pd.DataFrame,
    correct_intervention_list: List[str],
    use_retrieval_filter: bool = True,
    min_per_label: int = 3,
) -> DataContext:
    df = _normalize_df(df_ff)

    if use_retrieval_filter:
        df = df[df["intervention_name"].isin(correct_intervention_list)].copy()

    counts = df.groupby(["intervention_name", "label"]).size().unstack(fill_value=0)
    keep = counts[(counts.get(0, 0) >= min_per_label) & (counts.get(1, 0) >= min_per_label)].index.tolist()
    df_valid = df[df["intervention_name"].isin(keep)].copy()

    return DataContext(df_ff=df_valid, valid_interventions=sorted(keep), retrieval_used=use_retrieval_filter)

# ----------------------------
# Sampling helpers
# ----------------------------
def draw_random_intervention(valid_interventions: List[str]) -> str:
    return RNG.choice(valid_interventions)

def draw_random_label() -> int:
    return RNG.randint(0, 1)

def _sample_k(df: pd.DataFrame, intervention: str, label: int, k: int) -> List[Dict]:
    g = df[(df["intervention_name"] == intervention) & (df["label"] == label)]
    if len(g) == 0:
        # extremely rare due to filtering—but guard anyway
        g = df[df["label"] == label]
    if len(g) < k:
        g = g.sample(n=k, replace=True, random_state=RNG.randint(1, 10**9))
    else:
        g = g.sample(n=k, replace=False, random_state=RNG.randint(1, 10**9))
    return [{"text": row["text"], "label": int(row["label"])} for _, row in g.iterrows()]

def build_examples_for_variant(
    dc: DataContext, intervention: str, target_label: int, cfg: VariantConfig
) -> List[Tuple[str,int]]:
    if cfg.label_distribution == "mixed_head":
        opp = 1 - target_label
        rows_opp = _sample_k(dc.df_ff, intervention, opp, 1)
        rows_tar = _sample_k(dc.df_ff, intervention, target_label, N_EXAMPLES - 1)
        ex = [(r["text"], r["label"]) for r in (rows_opp + rows_tar)]
    elif cfg.label_distribution == "mixed_tail":
        opp = 1 - target_label
        rows_tar = _sample_k(dc.df_ff, intervention, target_label, N_EXAMPLES - 1)
        rows_opp = _sample_k(dc.df_ff, intervention, opp, 1)
        ex = [(r["text"], r["label"]) for r in (rows_tar + rows_opp)]
    else:
        rows_tar = _sample_k(dc.df_ff, intervention, target_label, N_EXAMPLES)
        ex = [(r["text"], r["label"]) for r in rows_tar]

    if cfg.example_ordering == "shuffled":
        RNG.shuffle(ex)
    return ex

def get_examples_with_budget(dc: DataContext, cfg: VariantConfig) -> Tuple[List[Tuple[str,int]], str, int]:
    for _ in range(1000):
        intervention = draw_random_intervention(dc.valid_interventions)
        target_label = draw_random_label()
        ex = build_examples_for_variant(dc, intervention, target_label, cfg)
        total = sum(num_tokens_from_string(t[0]) for t in ex)
        if total <= TOKEN_BUDGET_EXAMPLES:
            return ex, intervention, target_label
    # Fallback once
    intervention = draw_random_intervention(dc.valid_interventions)
    target_label = draw_random_label()
    ex = build_examples_for_variant(dc, intervention, target_label, cfg)
    return ex, intervention, target_label

# ----------------------------
# Prompt builders
# ----------------------------
def build_reasoning_messages(
    examples: List[Tuple[str,int]],
    intervention_name: str,
    target_label: int,
    cfg: VariantConfig
) -> List[dict]:
    lex = make_label_lexicon(cfg.class_name_scheme, target_label)

    chunks = []
    for text, lab in examples:
        head = make_label_lexicon(cfg.class_name_scheme, lab)["header_token"]
        chunks.append(f"{head} clinical trial of {intervention_name}: {text}")

    chunks.append(reason_format_prompt)
    chunks.append(_format_partial(
        reason_data_generation_prompt,
        name=intervention_name,
        intervention=intervention_name,
        drug=intervention_name,
        type=lex["type_noun"],
        label=lex["label_title"]
    ))
    chunks.append(reason_diversity_prompt)

    return [
        {"role": "system", "content": reason_context_prompt},
        {"role": "user", "content": " \n".join(chunks)},
    ]

def build_generation_messages(
    examples: List[Tuple[str,int]],
    intervention_name: str,
    target_label: int,
    cfg: VariantConfig,
    five_reasons: Optional[str] = None
) -> List[dict]:
    lex = make_label_lexicon(cfg.class_name_scheme, target_label)

    chunks = []
    if five_reasons is not None and str(five_reasons).strip():
        chunks.append(_format_partial(
            reason_prompt,
            type=lex["type_noun"],
            name=intervention_name,
            intervention=intervention_name,
            drug=intervention_name,
            label=lex["label_title"],
            reasons=five_reasons
        ))

    for text, lab in examples:
        head = make_label_lexicon(cfg.class_name_scheme, lab)["header_token"]
        chunks.append(f"{head} clinical trial of {intervention_name}: {text}")

    chunks.append(_format_partial(
        constraint_prompt,
        name=intervention_name,
        intervention=intervention_name,
        drug=intervention_name,
        type=lex["type_noun"],
        label=lex["label_title"]
    ))
    chunks.append(_format_partial(
        data_generation_prompt,
        type=lex["type_noun"],
        name=intervention_name,
        intervention=intervention_name,
        drug=intervention_name,
        label=lex["label_title"]
    ))
    chunks.append(diversity_prompt)

    return [
        {"role": "system", "content": context_prompt},
        {"role": "user", "content": " \n".join(chunks)},
    ]

# ----------------------------
# Model call
# ----------------------------
def call_openai(messages: List[dict], temperature: float = TEMPERATURE) -> str:
    out = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return out.choices[0].message.content

# ----------------------------
# Generate one trial (baseline/variant/ablation)
# ----------------------------
def generate_one_trial(
    dc: DataContext,
    cfg: VariantConfig,
    use_reasoning: bool = True
) -> Tuple[str, str, int]:
    examples, intervention_name, target_label = get_examples_with_budget(dc, cfg)

    if use_reasoning:
        msg_r = build_reasoning_messages(examples, intervention_name, target_label, cfg)
        five_reasons = call_openai(msg_r, temperature=TEMPERATURE)
        msg_g = build_generation_messages(examples, intervention_name, target_label, cfg, five_reasons=five_reasons)
    else:
        msg_g = build_generation_messages(examples, intervention_name, target_label, cfg, five_reasons=None)

    trial_text = call_openai(msg_g, temperature=TEMPERATURE)
    return trial_text, intervention_name, target_label

# ----------------------------
# Saving utilities
# ----------------------------
def _next_file_index(out_dir: str) -> int:
    pat = re.compile(r'^synthetic_(\d{6})\.txt$')
    max_idx = -1
    if os.path.isdir(out_dir):
        for fn in os.listdir(out_dir):
            m = pat.match(fn)
            if m:
                try:
                    idx = int(m.group(1))
                    max_idx = max(max_idx, idx)
                except ValueError:
                    pass
    return max_idx + 1

def run_and_save_variant(
    n: int,
    out_dir: str,
    df_ff: pd.DataFrame,
    correct_intervention_list: List[str],
    cfg: VariantConfig,
    labels_filename: str = "labels.txt",
    append_labels: bool = True,
    use_retrieval_filter: bool = True,
    use_reasoning: bool = True,
    min_per_label: int = 3
):
    os.makedirs(out_dir, exist_ok=True)
    labels_path = os.path.join(out_dir, labels_filename)

    dc = build_data_context(
        df_ff=df_ff,
        correct_intervention_list=correct_intervention_list,
        use_retrieval_filter=use_retrieval_filter,
        min_per_label=min_per_label
    )
    if len(dc.valid_interventions) == 0:
        raise RuntimeError("No valid interventions after filtering. Lower min_per_label or disable retrieval filter.")

    start_idx = _next_file_index(out_dir)
    mode = "a" if append_labels and os.path.exists(labels_path) else "w"

    with open(labels_path, mode, encoding="utf-8") as labf:
        idx = start_idx
        for _ in range(n):
            text, name, y = generate_one_trial(dc, cfg, use_reasoning=use_reasoning)
            with open(os.path.join(out_dir, f"synthetic_{idx:06d}.txt"), "w", encoding="utf-8") as f:
                f.write(text)
            labf.write(f"{int(y)}\n")
            idx += 1
        labf.flush()

# ----------------------------
# THREE SEPARATE RUNNERS
# ----------------------------

def run_baseline(
    n: int,
    out_dir: str,
    df_ff: pd.DataFrame,
    correct_intervention_list: List[str],
):
    """Baseline only: retrieval ✅, reasoning ✅, same-label, as-is, Success/Failure phrasing."""
    cfg = VariantConfig(label_distribution="same", example_ordering="as_is", class_name_scheme="success_failure")
    run_and_save_variant(
        n=n,
        out_dir=out_dir,
        df_ff=df_ff,
        correct_intervention_list=correct_intervention_list,
        cfg=cfg,
        use_retrieval_filter=True,
        use_reasoning=True
    )

def run_variants(
    n_per_variant: int,
    root_dir: str,
    df_ff: pd.DataFrame,
    correct_intervention_list: List[str],
):
    """
    Prompt-sensitivity variants only (retrieval ✅, reasoning ✅):
      - shuffled (same-label, shuffled ordering, success/failure)
      - mixed_head (as_is, success/failure)
      - mixed_tail (as_is, success/failure)
      - positive_negative (same-label, as_is)
    """
    os.makedirs(root_dir, exist_ok=True)
    variants = [
        VariantConfig("same",       "shuffled", "success_failure"),   # shuffled
        VariantConfig("mixed_head", "as_is",    "success_failure"),   # mixed_head
        VariantConfig("mixed_tail", "as_is",    "success_failure"),   # mixed_tail
        VariantConfig("same",       "as_is",    "positive_negative"), # positive_negative
    ]
    for cfg in variants:
        tag = f"{cfg.class_name_scheme}_{cfg.label_distribution}_{cfg.example_ordering}"
        out_dir = os.path.join(root_dir, tag)
        run_and_save_variant(
            n=n_per_variant,
            out_dir=out_dir,
            df_ff=df_ff,
            correct_intervention_list=correct_intervention_list,
            cfg=cfg,
            use_retrieval_filter=True,
            use_reasoning=True
        )

def run_ablations(
    n: int,
    root_dir: str,
    df_ff: pd.DataFrame,
    correct_intervention_list: List[str],
):
    """Ablations only (baseline config, no variants): no_retrieval and no_reasoning."""
    os.makedirs(root_dir, exist_ok=True)
    cfg = VariantConfig("same", "as_is", "success_failure")

    # 1) No retrieval (reasoning on)
    out_dir = os.path.join(root_dir, "no_retrieval")
    run_and_save_variant(
        n=n,
        out_dir=out_dir,
        df_ff=df_ff,
        correct_intervention_list=correct_intervention_list,
        cfg=cfg,
        use_retrieval_filter=False,
        use_reasoning=True
    )

    # 2) No reasoning (retrieval on)
    out_dir = os.path.join(root_dir, "no_reasoning")
    run_and_save_variant(
        n=n,
        out_dir=out_dir,
        df_ff=df_ff,
        correct_intervention_list=correct_intervention_list,
        cfg=cfg,
        use_retrieval_filter=True,
        use_reasoning=False
    )


In [ ]:
%rm -rf /content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants

In [ ]:
base_out = "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/baseline"
run_baseline(400, base_out, df_ff, list(correct_intervention_list))

In [ ]:
variants_root = "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/prompt_variants"
run_variants(400, variants_root, df_ff, list(correct_intervention_list))

In [ ]:
ablations_root = "/content/drive/MyDrive/ColabRepos/REPO/variants_ablations/ablations"
run_ablations(400, ablations_root, df_ff, list(correct_intervention_list))